# 7. Fine-tuning: does the backbone help on RBPs it never saw?

The 3M backbones were pretrained on **one split's training RBPs, all HepG2**. The split
also names the RBPs it held out, and grades them by how similar their binding is to the
training set, so "seen" and "unseen" here are recorded facts rather than guesses:

| tier | meaning | in this notebook |
|---|---|---|
| `seen` | in the backbone's pretraining tasks | PTBP1, RBFOX2 |
| `similar` | held out, but binds like something seen | QKI, HNRNPK |
| `different` | held out, lowest similarity to training | TRA2A, AGGF1 |

Every RBP appears twice, once for HepG2 and once for K562. **No K562 track was seen during
pretraining at all**, because there is no K562 backbone, so the K562 half is a second axis
of transfer on top of the RBP one.

The task is binary: does this 600 nt tile overlap an IDR peak? The labels were built from
the peak files in notebook 8, not taken from a prepared label matrix.

In [1]:
import numpy as np
import torch

torch.set_num_threads(2)
torch.manual_seed(0)

import parnet_tutorials as pt

MODEL = 'parnet-3m-pretrain-E2'

train = torch.load(pt.classification_path('peaks_train'), map_location='cpu', weights_only=False)
valid = torch.load(pt.classification_path('peaks_valid'), map_location='cpu', weights_only=False)

tracks = train['rbps']

# A workshop laptop runs this on CPU, and pushing 87k tiles through the backbone takes
# tens of minutes there. Sampling the validation split at random keeps each track's
# positive rate unchanged in expectation, so AUPRC stays comparable to its baseline;
# raise this if you have a GPU.
VALIDATION_TILES = 12_000

_rng = np.random.default_rng(1)
_keep = np.sort(_rng.choice(len(valid['names']), size=min(VALIDATION_TILES, len(valid['names'])), replace=False))
valid = {
    'seq': valid['seq'][torch.from_numpy(_keep)],
    'labels': valid['labels'][torch.from_numpy(_keep)],
    'names': [valid['names'][k] for k in _keep],
    'rbps': valid['rbps'],
}

print(f'{len(train["names"]):,} training tiles, {len(valid["names"]):,} validation tiles sampled')
print(f'{len(tracks)} tracks: {", ".join(tracks)}')

94,743 training tiles, 12,000 validation tiles sampled
12 tracks: AGGF1_HepG2, AGGF1_K562, HNRNPK_HepG2, HNRNPK_K562, PTBP1_HepG2, PTBP1_K562, QKI_HepG2, QKI_K562, RBFOX2_HepG2, RBFOX2_K562, TRA2A_HepG2, TRA2A_K562


## Which of these did the backbone actually see?

This is read off the model's own task axis rather than assumed, which is also how we catch
that a K562 track cannot have been seen by a HepG2-only backbone.

In [2]:
pretrained_tasks = set(pt.model_tasks(MODEL))

TIER = {
    'PTBP1': 'seen', 'RBFOX2': 'seen',
    'QKI': 'similar', 'HNRNPK': 'similar',
    'TRA2A': 'different', 'AGGF1': 'different',
}

rows = []
for i, track in enumerate(tracks):
    gene, cell_type = track.rsplit('_', 1)
    rows.append({
        'track': track,
        'tier': TIER[gene],
        'cell_type': cell_type,
        'seen_in_pretraining': track in pretrained_tasks,
        'train_positives': int(train['labels'][:, i].sum()),
    })

for r in rows:
    flag = 'yes' if r['seen_in_pretraining'] else 'no '
    print(f'  {r["track"]:16s} {r["tier"]:9s} seen={flag}  positives={r["train_positives"]:5d}')

print()
print('seen by the backbone:', sum(r['seen_in_pretraining'] for r in rows), 'of', len(rows))
print('...and every one of those is HepG2, because this backbone has no K562 task.')

  AGGF1_HepG2      different seen=no   positives=  164
  AGGF1_K562       different seen=no   positives=  465
  HNRNPK_HepG2     similar   seen=no   positives=  587
  HNRNPK_K562      similar   seen=no   positives=  358
  PTBP1_HepG2      seen      seen=yes  positives= 1134
  PTBP1_K562       seen      seen=no   positives= 1209
  QKI_HepG2        similar   seen=no   positives= 1330
  QKI_K562         similar   seen=no   positives=  424
  RBFOX2_HepG2     seen      seen=yes  positives=  651
  RBFOX2_K562      seen      seen=no   positives=  432
  TRA2A_HepG2      different seen=no   positives=  145
  TRA2A_K562       different seen=no   positives=  273

seen by the backbone: 2 of 12
...and every one of those is HepG2, because this backbone has no K562 task.


## Building a trainable subset

Positive rates are between 0.15 and 1.4 per cent. Sampling tiles uniformly would leave a
couple of dozen positives for the sparsest track and the metrics would be noise, so every
tile that is positive for **any** track is kept, plus a random sample of negatives.

That inflates the positive rate in training, which is fine, and would corrupt the
evaluation, which is not. **Validation is sampled uniformly instead**, never enriched, so
each track's positive rate is preserved in expectation and AUPRC stays comparable to its
baseline.

In [3]:
NEGATIVES_PER_POSITIVE = 2

labels = train['labels'].numpy()
is_positive = labels.any(axis=1)
positive_idx = np.flatnonzero(is_positive)
negative_idx = np.flatnonzero(~is_positive)

rng = np.random.default_rng(0)
sampled_negatives = rng.choice(
    negative_idx, size=min(len(negative_idx), NEGATIVES_PER_POSITIVE * len(positive_idx)), replace=False
)
subset = np.sort(np.concatenate([positive_idx, sampled_negatives]))

X_train = train['seq'][torch.from_numpy(subset)]
y_train = train['labels'][torch.from_numpy(subset)].float()

print(f'kept every positive tile: {len(positive_idx):,}')
print(f'plus {len(sampled_negatives):,} negatives -> {len(subset):,} training tiles')
print(f'validation sampled uniformly: {len(valid["names"]):,} tiles')

kept every positive tile: 5,510
plus 11,020 negatives -> 16,530 training tiles
validation sampled uniformly: 12,000 tiles


## Backbone plus a twelve-output head

One head over all twelve tracks, trained once. Training twelve separate models would say
nothing extra: the comparison we care about is between tracks, and sharing the backbone
holds everything except the track constant.

Only the head is trained. On a few thousand tiles that is the right default; unfreezing
more would mostly fit noise.

In [4]:
import torch.nn as nn

from parnet.finetuning import describe_trainable, set_trainable, stage_names
from parnet.models import RBPNet
from parnet.tokenization import ids_to_onehot

backbone = RBPNet.from_pretrained(pt.model_path(MODEL))
embedding_dim = backbone.describe()['body_width'] if hasattr(backbone, 'describe') else 320


class TileClassifier(nn.Module):
    """Pool the backbone's embedding over the window and score every track."""

    def __init__(self, backbone, n_tracks, embedding_dim):
        super().__init__()
        self.backbone = backbone
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Linear(embedding_dim, n_tracks)

    def forward(self, ids):
        embedding = self.backbone.embed(ids_to_onehot(ids))
        return self.head(self.pool(embedding).squeeze(-1))


model = TileClassifier(backbone, len(tracks), embedding_dim)
print('stages:', stage_names(backbone))
set_trainable(backbone, unfreeze_last=0)
print('trainable parameters:', describe_trainable(model))

stages: ['stem', 'body.0', 'body.1', 'body.2', 'body.3', 'body.4', 'body.5', 'body.6', 'body.7', 'body.8']
trainable parameters: {'n_trainable': 84159, 'n_frozen': 2789760, 'n_total': 2873919}


/home/l10n/projects/parnet-project/parnet--analyses--interpretation/.pixi/envs/parnet-dev-cu12/lib/python3.10/site-packages/torch/nn/modules/conv.py:370: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1025.)
  return F.conv1d(


## Training

A few epochs on a few thousand tiles. `pos_weight` compensates for the class imbalance
that survives the subsampling, per track.

In [5]:
from torch.utils.data import DataLoader, TensorDataset

EPOCHS = 3
positives = y_train.sum(dim=0).clamp(min=1.0)
pos_weight = ((len(y_train) - positives) / positives).clamp(max=50.0)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=1e-3)

loader = DataLoader(TensorDataset(X_train, y_train), batch_size=128, shuffle=True)
model.train()
for epoch in range(EPOCHS):
    total, seen = 0.0, 0
    for ids, target in loader:
        optimizer.zero_grad()
        loss = criterion(model(ids), target)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(ids)
        seen += len(ids)
    print(f'epoch {epoch + 1}: loss {total / seen:.4f}')

epoch 1: loss 0.9472


epoch 2: loss 0.8164


epoch 3: loss 0.7878


## Evaluation on the untouched validation split

AUPRC is reported next to the baseline, which is the track's own positive rate. On a task
this sparse, accuracy would read about 99 per cent for a model that predicts "no" every
time, and AUROC flatters sparse tasks too; the distance from AUPRC to its baseline is the
number that means something.

In [6]:
from sklearn.metrics import average_precision_score, roc_auc_score

model.eval()
scores = []
with torch.no_grad():
    for start in range(0, len(valid['seq']), 512):
        scores.append(model(valid['seq'][start:start + 512]))
scores = torch.cat(scores).numpy()
truth = valid['labels'].numpy()

for r in rows:
    column = tracks.index(r['track'])
    y_true, y_score = truth[:, column], scores[:, column]
    r['baseline_auprc'] = float(y_true.mean())
    r['auprc'] = float(average_precision_score(y_true, y_score))
    r['auroc'] = float(roc_auc_score(y_true, y_score))
    r['auprc_over_baseline'] = r['auprc'] / r['baseline_auprc']

print(f'{"track":16s} {"tier":10s} {"seen":5s} {"AUROC":>6s} {"AUPRC":>7s} {"base":>7s} {"x base":>7s}')
for r in sorted(rows, key=lambda r: (r['tier'], r['cell_type'])):
    print(f'{r["track"]:16s} {r["tier"]:10s} {"yes" if r["seen_in_pretraining"] else "no":5s} '
          f'{r["auroc"]:6.3f} {r["auprc"]:7.4f} {r["baseline_auprc"]:7.4f} {r["auprc_over_baseline"]:7.1f}')

track            tier       seen   AUROC   AUPRC    base  x base
AGGF1_HepG2      different  no     0.888  0.3432  0.0030   114.4
TRA2A_HepG2      different  no     0.937  0.0716  0.0024    29.6
AGGF1_K562       different  no     0.836  0.3110  0.0043    73.2
TRA2A_K562       different  no     0.947  0.1366  0.0018    78.1
PTBP1_HepG2      seen       yes    0.931  0.3293  0.0152    21.7
RBFOX2_HepG2     seen       yes    0.900  0.0842  0.0088     9.6
PTBP1_K562       seen       no     0.929  0.2529  0.0104    24.3
RBFOX2_K562      seen       no     0.826  0.0395  0.0037    10.8
HNRNPK_HepG2     similar    no     0.954  0.3114  0.0079    39.3
QKI_HepG2        similar    no     0.765  0.0632  0.0187     3.4
HNRNPK_K562      similar    no     0.902  0.1665  0.0032    52.6
QKI_K562         similar    no     0.696  0.0098  0.0041     2.4


## Seen against unseen

In [7]:
def summarise(selected, label):
    if not selected:
        return
    print(f'{label:34s} n={len(selected):2d}  '
          f'AUROC {np.mean([r["auroc"] for r in selected]):.3f}  '
          f'AUPRC x baseline {np.mean([r["auprc_over_baseline"] for r in selected]):5.1f}')


summarise([r for r in rows if r['seen_in_pretraining']], 'seen in pretraining')
summarise([r for r in rows if not r['seen_in_pretraining']], 'not seen in pretraining')
print()
for tier in ('seen', 'similar', 'different'):
    summarise([r for r in rows if r['tier'] == tier], f'tier: {tier}')
print()
for cell_type in ('HepG2', 'K562'):
    summarise([r for r in rows if r['cell_type'] == cell_type], f'cell type: {cell_type}')

seen in pretraining                n= 2  AUROC 0.915  AUPRC x baseline  15.7
not seen in pretraining            n=10  AUROC 0.868  AUPRC x baseline  42.8

tier: seen                         n= 4  AUROC 0.896  AUPRC x baseline  16.6
tier: similar                      n= 4  AUROC 0.830  AUPRC x baseline  24.4
tier: different                    n= 4  AUROC 0.902  AUPRC x baseline  73.8

cell type: HepG2                   n= 6  AUROC 0.896  AUPRC x baseline  36.3
cell type: K562                    n= 6  AUROC 0.856  AUPRC x baseline  40.2


## What to take away

Read the table before the summary. With twelve tracks, two of them carrying only a few
hundred positives, the per-tier averages rest on two or three numbers each and a single
sparse track can move them; `AGGF1` and `TRA2A` are the ones to be careful with.

What the setup does support is the comparison itself. Every track here shares one backbone,
one head architecture, one training run and one evaluation split, so differences between
tracks are not differences in treatment. The honest reading of any gap between tiers is
that it combines two things this notebook cannot separate: how much the pretrained
representation transfers, and how learnable each RBP is in the first place. A track the
21M also predicts poorly was probably never going to be easy.